*0.3 Classical NLP*

# Lemmatization

**The situation.** Product wants a dashboard of "top issues" built from the words in tickets. The stemmed version shows `charg`, `invoic`, `cancel` — not words a manager can read. And "better" stemmed is "better", never linked to "good"; "was" is never linked to "be".

**Lemmatization.** Reduce each word to its dictionary form (the *lemma*) using the word's part of speech and a vocabulary: charged → charge, invoices → invoice, better → well/good, was → be. Slower than stemming, always a real word. spaCy does it as part of its pipeline.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import spacy

nlp = spacy.load("en_core_web_sm")
text = "The customers were charged twice and the invoices were better handled after the refunds"
print(f"{'word':<12}{'lemma':>10}{'part of speech':>16}")
lemmas = []
for token in nlp(text):
    lemmas.append(token.lemma_)
    print(f"{token.text:<12}{token.lemma_:>10}{token.pos_:>16}")
assert "charge" in lemmas and "be" in lemmas and "well" in lemmas

word             lemma  part of speech
The                the             DET
customers     customer            NOUN
were                be             AUX
charged         charge            VERB
twice            twice             ADV
and                and           CCONJ
the                the             DET
invoices       invoice            NOUN
were                be             AUX
better            well             ADV
handled         handle            VERB
after            after             ADP
the                the             DET
refunds         refund            NOUN


**Reading the output.** "charged" → "charge", "were" → "be", "better" → "good", "invoices" → "invoice", "better" → "well" (here it is an adverb, so its lemma is "well"; as an adjective it would be "good"). Every lemma is a real word, chosen with the part of speech — a stemmer would leave "better" untouched.

**The dashboard, readable.** Count lemmas of nouns and verbs only.

In [3]:
from collections import Counter

tickets = [
    "I was charged twice for my invoices.",
    "Charging me twice is unacceptable, refund the invoice.",
    "Refunds for duplicated charges have not arrived.",
]
counts = Counter()
for document in nlp.pipe(tickets):  # nlp.pipe batches documents — use it for more than a handful
    for token in document:
        if token.pos_ in ("NOUN", "VERB"):
            counts[token.lemma_] += 1
print("top issues:", counts.most_common(4))
assert counts["charge"] >= 3 and counts["invoice"] >= 2

top issues: [('charge', 3), ('invoice', 2), ('refund', 2), ('duplicate', 1)]


**The rule to remember.** Lemmatize when the words will be read or must keep their meaning; stem when only matching matters and speed counts.

| Use it when | Don't when | Instead use |
|---|---|---|
| dashboards, topic analysis, features where "charge" and "charged" should merge | high-volume search where milliseconds matter | stemming |

**Watch out**
- Lemmas depend on the part of speech, and the tagger can be wrong on short or odd text ("refund" as noun vs verb).
- `en_core_web_sm` is small and fast; `_md`/`_lg` are more accurate. Pin the model version in `pyproject.toml`.
- Like stemming: not for text going into an LLM or embedding model.